### Database Design & Modeling — Complete Notes (OLTP + OLAP)

#### 1. Why Data Modeling?

Without data modeling, dumping all data into one giant table causes three major problems:

| Problem | Explanation |
|---|---|
| **Data Redundancy** | Repeating the same data (e.g., customer info) across rows. In the 1980s, storing 1 TB cost ~$300–400 million, so redundancy was extremely expensive. Even today with cheap storage, redundancy still causes downstream issues. |
| **Performance** | Querying a table with hundreds of unnecessary columns/rows degrades query speed even for simple lookups. |
| **Maintenance/Scalability** | Frequent updates (e.g., fixing a typo in a name) are hard to manage consistently without structure — leads to inconsistent data. |

**Data Modeling** = the technique of converting a big, messy, unstructured table into an **organized, structured** design (rows/columns organized into logical, purpose-driven tables).

---

#### 2. Layers of Data Modeling

| Layer | Also Called | What Happens |
|---|---|---|
| **Conceptual Layer** | Business layer | Requirement gathering — talking to business stakeholders/SMEs to understand needs. No technical work yet. |
| **Logical Layer** | — | Build the **ER diagram** (Entity-Relationship) — identify entities, attributes, relationships. This is the technical "logic" layer. |
| **Physical Layer** | Implementation layer | Convert the ER diagram into actual **SQL scripts** (CREATE TABLE, keys, constraints) and build the real database. |

> **AI note**: Conceptual and Logical layers are still primarily human-driven (business communication + design judgment). The Physical layer (writing SQL from an ER diagram) can now be **largely automated by AI/LLMs** — feed a screenshot of your ER diagram to an LLM and ask it to generate the SQL script (including keys/constraints); review before running.

---

#### 3. Types of Data Modeling: OLTP vs OLAP

| | OLTP (Online Transactional Processing) | OLAP (Online Analytical Processing) |
|---|---|---|
| Also called | Database modeling / operational database | Data warehousing |
| Purpose | Fast, frequent, reliable **transactions** (insert/update/delete) | **Reporting & analytics** / insights |
| Users | End customers/application users | Internal business users (managers, analysts, CEOs) |
| Data volume per operation | Small — usually 1 record or a few at a time | Huge — millions of records retrieved at once |
| Priority | **Speed of writes** (DML: insert/update/delete) | **Speed of reads/aggregation** over large data |
| Example | Amazon order placement, address change, wishlist | Sales dashboard, quarterly revenue report |
| Design style | Normalized | Denormalized (star/snowflake schema) |

> OLTP must be mastered first — OLAP modeling is built **on top of** an OLTP source (or equivalent raw data).

---

#### 4. OLTP Modeling — Step by Step

##### Step 1: Requirement Gathering (Conceptual Layer)

- Talk to stakeholders: business folks, SMEs, team leads, developers/BI analysts.
- Business people often give vague answers ("just build it") — **you must ask drilling questions**.
- Start by asking about **KPIs** / dashboard goals — this often reveals the real requirements.
- From answers, extract three things:
  - **Subject** → usually becomes a **table**
  - **Characteristic** → becomes a **column**, OR becomes its own **table (object)** if it can be further broken down (e.g., "orders" is a characteristic of "customers" but can be broken into its own table since it has sub-attributes like product, amount)
  - **Relation** → determined later, based on follow-up questions like "can a customer place multiple orders?"
- Two common requirement-gathering methodologies:
  - **Waterfall**: sequential — gather requirements → build ER diagram → build final product → show stakeholders (risk: they only see the end result and may reject it, causing rework). Not ideal.
  - **Agile**: iterative loops with stakeholders involved throughout (like scrum) — preferred industry standard.

##### Step 2: Logical Layer — Build Entities

- Convert each requirement's **subject** into an **entity** (table).
- Each entity gets an **ID / identifier column** (e.g., `customer_id`, `order_id`) — standard practice for uniquely identifying and later joining records.
- Example entities for an "Online Superstore": `customers`, `orders`, `payments`, `products`, `shipments`.

##### Step 3: Build the ER Diagram

- Tool used: **draw.io** (free, industry-common; Microsoft Visio is an alternative).
- Add entities as tables, list attributes (columns) under each.

##### Step 4: Data Types — key categories:

| Category | Types | Notes |
|---|---|---|
| Numeric | `INTEGER` (~±2 billion range), `BIGINT` (larger range, used for huge datasets, common in OLAP) | For whole numbers |
| Numeric (decimal) | `FLOAT` (32-bit, ~7 decimal digits, approximate), `DOUBLE` (64-bit, ~15–16 decimal digits, more precise), `NUMERIC(P,S)` (exact — P = total digits, S = digits after decimal) | Use `NUMERIC` for currency/financial data where exact precision matters |
| String | `CHAR(n)` (fixed length, pads to n), `VARCHAR(n)` (variable length up to n, saves space), `TEXT` (large unbounded text), `NVARCHAR` (Unicode/international characters) | Use `CHAR` for fixed-length codes (e.g., country codes), `VARCHAR` for variable text (e.g., email) |
| Date/Time | `DATE`, `TIME`, `TIMESTAMP` (date + time combined) | |
| Other | `BOOLEAN`, `JSONB` (some DBs), `UUID` (not universal) | Check specific DB docs |

---

#### 5. Keys — Core Concepts

| Key Type | Definition |
|---|---|
| **Primary Key (PK)** | Uniquely identifies each row; must be **unique** and **NOT NULL**; only **one PK per table**. |
| **Candidate Key** | Any column (or column combination) that *could* serve as the PK (unique + not null). A table can have multiple candidate keys. |
| **Alternate Key** | The candidate key(s) **not chosen** as the primary key. |
| **Super Key** | **Any** combination of columns that can uniquely identify a row (includes redundant/larger combinations, not just minimal ones). |
| **Unique Key** | Like a primary key, but **allows one NULL value**. |
| **Foreign Key (FK)** | A column in the **child table** that references the **primary key of the parent table** — creates the relationship/bridge between tables. Always flows **child → parent**. No uniqueness constraint required. |
| **Composite Primary Key** | When no single column is unique enough — a **combination of 2+ columns** together uniquely identifies a row (e.g., `order_id + product_id` in a junction table). |

**Relationship hierarchy** (memorize this):
- **Super Key** ⊇ **Candidate Key(s)** ⊇ **Primary Key** (one chosen) + **Alternate Key(s)** (the rest)

---

#### 6. Relationships & Cardinality

Cardinality defines how records in one entity relate to records in another. Three cardinality symbols (Crow's Foot notation — industry standard, preferred over Chen notation which is rarely used in practice):

| Symbol | Meaning |
|---|---|
| **0** (circle) | Zero — optional (minimum) |
| **1** (single line) | Exactly one |
| **Many** (crow's foot) | Many |

- Notation reads as: **outer symbol = maximum**, **inner symbol = minimum**.
- Always evaluate relationship **direction-aware**: "Orders → Customers" vs "Customers → Orders" can look different depending on which side you're viewing from.

##### Types of Relationships

| Type | Example | Notes |
|---|---|---|
| **One-to-One (1:1)** | Department ↔ Manager (one manager manages exactly one department) | Rare in practice |
| **One-to-Many / Many-to-One** | Customer → Orders (one customer can place many orders; one order belongs to one customer) | Most common |
| **Many-to-Many (M:N)** | Orders ↔ Products (one order can have many products; one product can appear in many orders) | **Cannot be implemented directly in a relational DB** — requires a **junction/bridge table** |

##### Resolving Many-to-Many
- Create a **bridge entity** (e.g., `order_items`) between the two tables.
- The bridge table holds the FKs from both sides (e.g., `order_id`, `product_id`) and typically uses a **composite primary key** of those two FKs.
- This converts the M:N relationship into **two 1:N relationships**: Orders → Order_Items (1:many) and Products → Order_Items (1:many).

---

#### 7. Normalization

**Goal**: Reduce data redundancy (definition: "database normalization is a process aimed at minimizing data redundancy").

Normal forms are **iterative/cumulative** — to be in 2NF, a table must already satisfy 1NF; to be in 3NF, it must satisfy 2NF, etc. Industry standard: aim for **at least 3NF** in ~90% of cases (10% may intentionally denormalize for performance, since storage is cheap today).

##### 1NF (First Normal Form)
Rules:
1. Table must have a **primary key** (each row uniquely identifiable).
2. Every column must hold **atomic (single) values** — no repeating groups/lists in a single cell.

Fix: "Explode" multi-value columns into separate rows. This often converts a single-column PK into a **composite PK**.

##### 2NF (Second Normal Form)
Rules:
1. Must already be in 1NF.
2. **No partial dependency** — every non-key column must depend on the **entire composite primary key**, not just part of it.

Fix: If a column depends on only part of the composite key (e.g., `customer_name` depends only on `order_id`, not on `order_id + product_id`), move it to a separate table.

##### 3NF (Third Normal Form)
Rules:
1. Must already be in 2NF.
2. **No transitive dependency** — non-key columns must depend **only** on the primary key, not on another non-key column.

Example: If `category_name` depends on `category_id` (a non-key column) rather than directly on the primary key, that's a transitive dependency — split into a separate `category` table.

##### BCNF (Boyce-Codd Normal Form)
- Stricter version of 3NF; removes **functional dependency** anomalies.
- Rule: for any functional dependency (X → Y), **X must be a super key**.
- Example: if `instructor` determines `course` (1 instructor teaches exactly 1 course) but `instructor` isn't a super key of the table, it violates BCNF → split into separate tables.
- **Not always applied in industry** — it's a design choice; 3NF is generally the accepted "good enough" standard today given cheap storage. Rarely goes beyond BCNF (4NF, 5NF, 6NF exist but are very rarely used in practice).

---

#### 8. Physical Layer — Implementation (with AI)

- Once the ER diagram (logical layer) is finalized, write SQL DDL (`CREATE TABLE`, constraints, keys) to implement it — this is the **physical layer**.
- **Modern approach**: feed a screenshot/image of the ER diagram to a multimodal LLM (e.g., GPT-5+) and ask it to generate the SQL script for a specific database (MySQL, Postgres, etc.).
  - The AI can infer table structure, data types, relationships (1:1, 1:N, M:N), and **foreign key constraints** automatically.
  - Saves significant time (a task that might take a data professional hours can be done in seconds) — but always **review the AI-generated script** before running it.

---

#### 9. OLAP Modeling — Building the Data Warehouse

OLAP modeling is built **on top of** the OLTP source, using a **Medallion Architecture**:

| Layer | Purpose |
|---|---|
| **Bronze (Raw)** | Exact, unmodified replica of source data, loaded **incrementally** (not full reload each time) — batch, hourly, or real-time depending on architecture (Lambda, Kappa, etc.) |
| **Silver** | Data is **transformed, cleaned, and aggregated**. Modern best practice: build a **OBT (One Big Table)** — join/combine all relevant source tables into a single wide table. Increasingly popular due to cheaper compute. |
| **Gold** | Break the OBT down into a proper **dimensional data model** (fact & dimension tables) — this is where **denormalized** star/snowflake schema modeling happens. |

> Note: OLAP data models are **intentionally denormalized** (opposite of OLTP normalization) — because OLAP prioritizes **query/read performance** over storage efficiency or write-side redundancy concerns. Storage is cheap; slow dashboards are not acceptable.

---

#### 10. Fact Tables vs Dimension Tables

| | Dimension Table | Fact Table |
|---|---|---|
| Focus | **Context** / theme (descriptive, textual data) | **Numbers that can be aggregated** (measures) |
| Contains | Names, categories, statuses, descriptive attributes + IDs | Amounts, quantities, prices paid, counts — anything summable/averageable |
| Example columns | `customer_name`, `email`, `product_name`, `category`, `payment_method`, `shipment_status` | `order_amount`, `quantity`, `line_amount`, `payment_amount` |
| Quantity per model | Typically **many** dimension tables | Typically **fewer** fact tables (often just one, but can have multiple for different business processes/goals — e.g., a separate fact table for cancellations) |
| Key test | "Can I aggregate (SUM/AVG/MIN/MAX) this column?" → If yes, it does NOT belong in a dimension table (even if numeric, e.g., `product_id` or `price` as a static/context value can still be a dimension attribute if it's not meant to be summed for business meaning — judge by intent, not just data type) |

##### Schema Types

| Schema | Structure | Notes |
|---|---|---|
| **Star Schema** | One central fact table directly connected to multiple dimension tables (no further branching) | Simplest, most common in modern data warehouses; best query performance |
| **Snowflake Schema** | A dimension table is further normalized into sub-dimension tables (branching out, like a snowflake) | More complex to maintain, extra joins hurt performance — **rarely used in modern solutions** since storage is cheap and performance is prioritized |

##### Building a Star Schema (Practical Steps)
1. Start from the **OBT** (One Big Table).
2. Identify contextual "themes" → create dimension tables first (e.g., `dim_customers`, `dim_products`, `dim_branch`, `dim_payments`, `dim_shipments`, `dim_date`).
3. Remaining aggregatable numeric columns → go into the **fact table** (e.g., `fact_sales` / `fact_orders`) along with the relevant foreign keys (IDs) from each dimension.
4. Join fact ↔ dimensions using the shared **IDs** or **surrogate keys**.

##### Surrogate Keys
- A simplified, auto-incrementing replacement (1, 2, 3…) for a natural/business key (e.g., a long order ID).
- Improves join performance and readability in the fact table.
- Not mandatory — a design choice for convenience.

---

#### 11. Slowly Changing Dimensions (SCD)

Dimension data changes over time — SCD types define **how to handle those changes**.

| Type | Behavior | Example Use Case |
|---|---|---|
| **Type 0** | **No changes allowed** — value is retained as originally recorded forever | Date of birth, original credit score |
| **Type 1** | **Overwrite / Upsert** — old value is simply updated in place; **no history retained** | Correcting a typo, simple attribute update; most commonly used in practice |
| **Type 2** | **Retain full history** — insert a **new row** for the change, keep the old row marked as expired using `start_date` / `end_date` columns (or a version flag). Old record gets an end date; new record gets start date with an open-ended end date (e.g., `9999-01-01` or NULL) | When historical accuracy matters (e.g., tracking what a supplier's state was on a given past date) — my personal favorite per the instructor |
| **Type 3** | Add a **new column** to store only the **previous value** (limited history — just one prior state, not full history) | Rarely used — considered a middle-ground with limited practical benefit |
| **Type 4** (rarer) | Maintain a **separate history table** alongside the main (Type 1) table | Not very practical — Type 2 usually preferred instead |
| **Type 5 / Type 6** (rarer) | Hybrid/combined approaches (e.g., Type 6 = combination of Type 1 + 2 + 3) | Rarely implemented in real-world data engineering |

> Industry practice: **Type 0, 1, 2, and 3** cover ~99% of real-world needs; Type 2 is especially important and commonly asked about in interviews. Implementation is typically done via **PySpark/Spark SQL MERGE statements** (similar logic to upsert) or tools like **dbt** on platforms like Databricks/Snowflake.

---

#### 12. Quick-Reference Cheat Sheet (Interview Recall)

- **3 Layers of Data Modeling**: Conceptual (requirements) → Logical (ER diagram) → Physical (SQL implementation).
- **OLTP** = transactional, fast writes, small data per operation, normalized. **OLAP** = analytical, fast reads, huge data volume, denormalized.
- **Keys**: Super Key ⊇ Candidate Keys ⊇ Primary Key + Alternate Keys. Foreign Key always lives in the **child** table and points to the **parent's** primary key.
- **Cardinality**: 0 (optional), 1 (exactly one), Many. Crow's Foot notation is the industry standard.
- **Many-to-Many** relationships require a **junction/bridge table** (composite PK of both FKs) — cannot be modeled directly.
- **Normalization order**: 1NF (atomic values + PK) → 2NF (no partial dependency) → 3NF (no transitive dependency) → BCNF (every determinant is a super key). Aim for at least 3NF.
- **Medallion Architecture for OLAP**: Bronze (raw incremental dump) → Silver (transformed/OBT) → Gold (dimensional model: fact + dimension tables).
- **Dimension tables** = context/text/descriptive attributes. **Fact tables** = aggregatable numeric measures. Test: "Can this column be summed/averaged meaningfully?" → Fact table; else → Dimension.
- **Star Schema** (simple, fast, modern standard) vs **Snowflake Schema** (normalized dimensions, more joins, rarely used today).
- **SCD Types**: Type 0 = never changes. Type 1 = overwrite (no history). Type 2 = new row + start/end dates (full history — most robust). Type 3 = one extra column for previous value only (limited history).
- **AI use case**: Feed ER diagram screenshot to an LLM → auto-generate SQL DDL with keys/constraints — review before executing.